<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula04d%20-%20GridSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [215]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [323]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

params = {'n_neighbors': range(1, 21, 2)}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy')
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

{'n_neighbors': 1}
0.7613300492610836


In [324]:
y_pred = grid.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.6111111111111112


In [334]:
from sklearn.model_selection import KFold
params = {
    'n_neighbors': range(1, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

{'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
0.8032019704433498


In [335]:
from sklearn.model_selection import RepeatedKFold

model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy',
                    cv=RepeatedKFold(n_splits=5, n_repeats=10))
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

{'metric': 'manhattan', 'n_neighbors': 1, 'weights': 'uniform'}
0.8015270935960591


In [347]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(KNeighborsClassifier(), X_train, y_train,
                         cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.51724138 0.75862069 0.67857143 0.60714286 0.85714286]
0.683743842364532


In [348]:
params = {
    'n_neighbors': range(1, 21, 2),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
model = KNeighborsClassifier()
grid = GridSearchCV(model, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))
scores = cross_val_score(grid, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.86206897 0.75862069 0.78571429 0.82142857 0.82142857]
0.8098522167487683


In [349]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
  ('scaler', StandardScaler()),
  ('model', grid)
])
scores = cross_val_score(pipeline, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[1.         0.96551724 0.96428571 0.96428571 0.89285714]
0.9573891625615765


In [351]:
params = {
    'model__n_neighbors': range(1, 21, 2),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

pipeline = Pipeline([
  ('scaler', StandardScaler()),
  ('model', KNeighborsClassifier())
])
grid = GridSearchCV(pipeline, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))
scores = cross_val_score(grid, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.96551724 1.         0.96428571 0.96428571 0.96428571]
0.9716748768472907


In [352]:
params = {
    'scaler__with_mean': [True, False],
    'scaler__with_std': [True, False],
    'model__n_neighbors': range(1, 21, 2),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

pipeline = Pipeline([
  ('scaler', StandardScaler()),
  ('model', KNeighborsClassifier())
])
grid = GridSearchCV(pipeline, params, scoring='accuracy', cv=KFold(n_splits=5, shuffle=True))
scores = cross_val_score(grid, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[1.         0.93103448 1.         1.         0.96428571]
0.9790640394088669


In [359]:
from sklearn.model_selection import RandomizedSearchCV

grid = RandomizedSearchCV(pipeline, params, scoring='accuracy',
                          cv=KFold(n_splits=5, shuffle=True), n_iter=20)
scores = cross_val_score(grid, X_train, y_train, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[1.         0.96551724 0.92857143 1.         0.89285714]
0.9573891625615765


In [361]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.5 MB/s eta 0:00:00


In [364]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

def objective(trial):
    # Define hyperparameters for StandardScaler
    with_mean = trial.suggest_categorical('scaler__with_mean', [True, False])
    with_std = trial.suggest_categorical('scaler__with_std', [True, False])

    # Define hyperparameters for KNeighborsClassifier
    n_neighbors = trial.suggest_int('model__n_neighbors', 1, 20, step=2)
    weights = trial.suggest_categorical('model__weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('model__metric', ['euclidean', 'manhattan', 'minkowski'])

    pipeline = Pipeline([
        ('scaler', StandardScaler(with_mean=with_mean, with_std=with_std)),
        ('model', KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, metric=metric))
    ])

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')

    return np.mean(scores)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # You can adjust n_trials as needed

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2026-04-08 00:47:33,008] A new study created in memory with name: no-name-8f4b3baa-09c6-4da7-8eac-0841f66dad66
/tmp/ipykernel_18898/2138879417.py:14: UserWarning: The distribution is specified by [1, 20] and step=2, but the range is not divisible by `step`. It will be replaced with [1, 19].
  n_neighbors = trial.suggest_int('model__n_neighbors', 1, 20, step=2)
[I 2026-04-08 00:47:33,038] Trial 0 finished with value: 0.8100985221674877 and parameters: {'scaler__with_mean': False, 'scaler__with_std': False, 'model__n_neighbors': 1, 'model__weights': 'uniform', 'model__metric': 'manhattan'}. Best is trial 0 with value: 0.8100985221674877.
/tmp/ipykernel_18898/2138879417.py:14: UserWarning: The distribution is specified by [1, 20] and step=2, but the range is not divisible by `step`. It will be replaced with [1, 19].
  n_neighbors = trial.suggest_int('model__n_neighbors', 1, 20, step=2)
[I 2026-04-08 00:47:33,066] Trial 1 finished with value: 0.7753694581280788 and parameters: {'scaler_

Number of finished trials:  10
Best trial:
  Value:  0.9788177339901478
  Params: 
    scaler__with_mean: True
    scaler__with_std: True
    model__n_neighbors: 5
    model__weights: uniform
    model__metric: euclidean
